# Day 078 | PINNs for Geotechnical Problems

## 100 Days of ML for Geotechnical Engineering
**Phase:** Deep Learning | **Author:** Ripon Chandra Malo | **Date:** YYYY-MM-DD

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| Physics-Informed Neural Network | NN trained with physics loss | Enforcing Darcy's law in seepage prediction |
| PDE residual loss | Penalize violation of governing equations | Laplace equation residual in flow domain |
| Boundary conditions | Constrain solution at boundaries | Fixed head upstream/downstream of dam |
| Collocation points | Random domain points for physics evaluation | Points sampled under the dam |
| Analytical comparison | Validate PINN against known solution | 1D/2D confined flow exact solutions |
| Automatic differentiation | Compute derivatives through the network | torch.autograd for partial derivatives |

---

## Our Geotechnical Problem

We want to solve **2D steady-state seepage under a dam** using a Physics-Informed Neural Network.

The governing equation is the **Laplace equation** (from Darcy's law for homogeneous, isotropic soil):

$$\nabla^2 h = \frac{\partial^2 h}{\partial x^2} + \frac{\partial^2 h}{\partial y^2} = 0$$

```
  Upstream (h=10m)       DAM        Downstream (h=2m)
  ──────────────── ▓▓▓▓▓▓▓▓▓▓▓▓▓▓ ────────────────
  ~~~~~~~~~~~~~~~~ ▓▓▓▓▓▓▓▓▓▓▓▓▓▓ ~~~~~~~~~~~~~~~~
  │                                               │
  │         Seepage domain (soil)                  │
  │         nabla^2 h = 0                          │
  │                                               │
  └───────────────────────────────────────────────┘
                  Impervious base (dh/dy = 0)
```

Instead of finite elements, we train a neural network to satisfy the PDE everywhere in the domain.

## Part 1: Import Libraries and Set Up Domain

In [ ]:
# ───────────────────────────────────────────────────────
# IMPORTS AND CONFIGURATION
# ───────────────────────────────────────────────────────

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Domain definition: rectangular region under the dam
# x in [0, L], y in [0, D]  (L = length, D = depth)
L = 10.0   # domain length (m)
D = 5.0    # domain depth (m)

# Boundary conditions
h_upstream   = 10.0   # hydraulic head upstream (m)
h_downstream = 2.0    # hydraulic head downstream (m)

print(f"Domain: {L}m x {D}m")
print(f"Upstream head:   {h_upstream} m")
print(f"Downstream head: {h_downstream} m")
print(f"Head difference: {h_upstream - h_downstream} m")

---

## Part 2: Generate Collocation and Boundary Points

PINNs need two types of training points:
1. **Collocation points** — random interior points where the PDE must be satisfied
2. **Boundary points** — points on the boundary where BCs are enforced

In [ ]:
# ───────────────────────────────────────────────────────
# GENERATE TRAINING POINTS
# ───────────────────────────────────────────────────────

def generate_collocation_points(n_interior, n_boundary):
    """Generate interior collocation points and boundary points."""
    
    # Interior points (random sampling in the domain)
    x_int = torch.rand(n_interior, 1) * L
    y_int = torch.rand(n_interior, 1) * D
    xy_interior = torch.cat([x_int, y_int], dim=1).to(device)
    
    # Boundary points
    n_per_side = n_boundary // 4
    
    # Left boundary (x=0): h = h_upstream
    x_left = torch.zeros(n_per_side, 1)
    y_left = torch.rand(n_per_side, 1) * D
    xy_left = torch.cat([x_left, y_left], dim=1).to(device)
    h_left = torch.full((n_per_side, 1), h_upstream).to(device)
    
    # Right boundary (x=L): h = h_downstream
    x_right = torch.full((n_per_side, 1), L)
    y_right = torch.rand(n_per_side, 1) * D
    xy_right = torch.cat([x_right, y_right], dim=1).to(device)
    h_right = torch.full((n_per_side, 1), h_downstream).to(device)
    
    # Top boundary (y=D): h varies linearly (simplified)
    x_top = torch.rand(n_per_side, 1) * L
    y_top = torch.full((n_per_side, 1), D)
    xy_top = torch.cat([x_top, y_top], dim=1).to(device)
    h_top = (h_upstream + (h_downstream - h_upstream) * x_top / L).to(device)
    
    # Bottom boundary (y=0): impervious — dh/dy = 0 (Neumann)
    x_bot = torch.rand(n_per_side, 1) * L
    y_bot = torch.zeros(n_per_side, 1)
    xy_bottom = torch.cat([x_bot, y_bot], dim=1).to(device)
    
    # Combine Dirichlet boundary data
    xy_bc = torch.cat([xy_left, xy_right, xy_top], dim=0)
    h_bc  = torch.cat([h_left, h_right, h_top], dim=0)
    
    return xy_interior, xy_bc, h_bc, xy_bottom

n_interior = 1000
n_boundary = 400

xy_int, xy_bc, h_bc, xy_bot = generate_collocation_points(n_interior, n_boundary)

print(f"Interior collocation points: {xy_int.shape[0]}")
print(f"Dirichlet boundary points:   {xy_bc.shape[0]}")
print(f"Neumann boundary points:     {xy_bot.shape[0]}")

# Visualize points
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.scatter(xy_int[:, 0].cpu(), xy_int[:, 1].cpu(), s=1, alpha=0.3, label='Interior')
ax.scatter(xy_bc[:, 0].cpu(), xy_bc[:, 1].cpu(), s=10, c='red', label='Dirichlet BC')
ax.scatter(xy_bot[:, 0].cpu(), xy_bot[:, 1].cpu(), s=10, c='green', label='Neumann BC')
ax.set_xlabel('x (m)', fontsize=12)
ax.set_ylabel('y (m)', fontsize=12)
ax.set_title('Collocation Points for PINN Training', fontsize=13)
ax.legend(fontsize=10)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## Part 3: Define the PINN Architecture

The PINN is a standard feedforward network that takes (x, y) coordinates and outputs hydraulic head h.
The key difference from a regular NN: we add a **physics loss** based on the PDE residual.

```
  Input (x, y)  ->  [64] -> [64] -> [64] -> [64]  ->  Output h(x,y)
     (2)          tanh    tanh    tanh    tanh        (1)
```

In [ ]:
# ───────────────────────────────────────────────────────
# PINN MODEL DEFINITION
# ───────────────────────────────────────────────────────

class SeepagePINN(nn.Module):
    """Physics-Informed Neural Network for 2D steady-state seepage."""
    
    def __init__(self, hidden_size=64, n_layers=4):
        super().__init__()
        
        layers = []
        layers.append(nn.Linear(2, hidden_size))  # input: (x, y)
        layers.append(nn.Tanh())
        
        for _ in range(n_layers - 1):
            layers.append(nn.Linear(hidden_size, hidden_size))
            layers.append(nn.Tanh())
        
        layers.append(nn.Linear(hidden_size, 1))  # output: h
        
        self.network = nn.Sequential(*layers)
        
        # Xavier initialization (good for tanh)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)
    
    def forward(self, xy):
        return self.network(xy)

model = SeepagePINN(hidden_size=64, n_layers=4).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"PINN architecture:")
print(model)
print(f"\nTotal parameters: {total_params:,}")

---

## Part 4: Physics Loss — The Heart of PINNs

The physics loss enforces the Laplace equation:

$$\mathcal{L}_{\text{physics}} = \frac{1}{N} \sum_{i=1}^{N} \left( \frac{\partial^2 h}{\partial x^2} + \frac{\partial^2 h}{\partial y^2} \right)^2$$

We compute second derivatives using **automatic differentiation** (`torch.autograd.grad`).

In [ ]:
# ───────────────────────────────────────────────────────
# PHYSICS LOSS: LAPLACE EQUATION RESIDUAL
# ───────────────────────────────────────────────────────

def compute_pde_residual(model, xy):
    """
    Compute the Laplace equation residual: d2h/dx2 + d2h/dy2 = 0
    
    Uses torch.autograd.grad for automatic differentiation.
    """
    xy.requires_grad_(True)
    
    h = model(xy)  # predict head
    
    # First derivatives: dh/dx, dh/dy
    dh = torch.autograd.grad(
        h, xy, 
        grad_outputs=torch.ones_like(h),
        create_graph=True  # need graph for second derivatives
    )[0]
    
    dh_dx = dh[:, 0:1]  # dh/dx
    dh_dy = dh[:, 1:2]  # dh/dy
    
    # Second derivatives: d2h/dx2, d2h/dy2
    d2h_dx2 = torch.autograd.grad(
        dh_dx, xy,
        grad_outputs=torch.ones_like(dh_dx),
        create_graph=True
    )[0][:, 0:1]
    
    d2h_dy2 = torch.autograd.grad(
        dh_dy, xy,
        grad_outputs=torch.ones_like(dh_dy),
        create_graph=True
    )[0][:, 1:2]
    
    # Laplace residual: should be zero
    residual = d2h_dx2 + d2h_dy2
    
    return residual, dh_dy

# Test with a small batch
test_xy = xy_int[:10].clone()
res, _ = compute_pde_residual(model, test_xy)
print(f"PDE residual shape: {res.shape}")
print(f"Initial residual magnitude: {res.abs().mean().item():.6f}")
print("(This will decrease during training)")

---

## Part 5: Combined Loss Function

The total PINN loss combines three terms:

- **Physics loss**: PDE residual at interior points
- **BC loss**: MSE at Dirichlet boundary points
- **Neumann loss**: dh/dy = 0 at the impervious base

In [ ]:
# ───────────────────────────────────────────────────────
# COMBINED LOSS FUNCTION
# ───────────────────────────────────────────────────────

def pinn_loss(model, xy_interior, xy_bc, h_bc, xy_neumann,
              w_physics=1.0, w_bc=10.0, w_neumann=1.0):
    """
    Compute total PINN loss.
    
    Args:
        w_physics: weight for PDE residual loss
        w_bc:      weight for Dirichlet BC loss (higher = stricter enforcement)
        w_neumann: weight for Neumann BC loss
    """
    # 1. Physics loss (Laplace equation)
    residual, _ = compute_pde_residual(model, xy_interior)
    loss_physics = torch.mean(residual ** 2)
    
    # 2. Dirichlet BC loss
    h_pred_bc = model(xy_bc)
    loss_bc = torch.mean((h_pred_bc - h_bc) ** 2)
    
    # 3. Neumann BC loss (dh/dy = 0 at bottom)
    _, dh_dy_bot = compute_pde_residual(model, xy_neumann)
    loss_neumann = torch.mean(dh_dy_bot ** 2)
    
    # Total weighted loss
    loss_total = w_physics * loss_physics + w_bc * loss_bc + w_neumann * loss_neumann
    
    return loss_total, loss_physics.item(), loss_bc.item(), loss_neumann.item()

print("Loss function defined with three components:")
print("  1. Physics loss  — enforces Laplace equation")
print("  2. BC loss       — enforces Dirichlet boundaries")
print("  3. Neumann loss  — enforces impervious base")

---

## Part 6: Train the PINN

In [ ]:
# ───────────────────────────────────────────────────────
# TRAINING LOOP
# ───────────────────────────────────────────────────────

# Re-initialize model
model = SeepagePINN(hidden_size=64, n_layers=4).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=500, gamma=0.5)

n_epochs = 1500
history = {'total': [], 'physics': [], 'bc': [], 'neumann': []}

print(f"Training PINN for {n_epochs} epochs...")
print(f"{'Epoch':>6} {'Total':>10} {'Physics':>10} {'BC':>10} {'Neumann':>10}")
print("─" * 50)

for epoch in range(n_epochs):
    optimizer.zero_grad()
    
    # Resample interior points every 500 epochs for better coverage
    if epoch % 500 == 0:
        xy_int, xy_bc, h_bc, xy_bot = generate_collocation_points(n_interior, n_boundary)
    
    loss_total, l_phys, l_bc, l_neum = pinn_loss(
        model, xy_int, xy_bc, h_bc, xy_bot
    )
    
    loss_total.backward()
    optimizer.step()
    scheduler.step()
    
    history['total'].append(loss_total.item())
    history['physics'].append(l_phys)
    history['bc'].append(l_bc)
    history['neumann'].append(l_neum)
    
    if epoch % 500 == 0 or epoch == n_epochs - 1:
        print(f"{epoch:6d} {loss_total.item():10.6f} {l_phys:10.6f} {l_bc:10.6f} {l_neum:10.6f}")

print(f"\nTraining complete!")

---

## Part 7: Visualize Training History

In [ ]:
# ───────────────────────────────────────────────────────
# LOSS HISTORY VISUALIZATION
# ───────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Total loss
axes[0].semilogy(history['total'], 'b-', linewidth=1.5, label='Total')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss (log scale)', fontsize=12)
axes[0].set_title('Total PINN Loss', fontsize=13)
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=11)

# Component losses
axes[1].semilogy(history['physics'], 'r-', linewidth=1.2, alpha=0.8, label='Physics (PDE)')
axes[1].semilogy(history['bc'], 'g-', linewidth=1.2, alpha=0.8, label='Boundary (Dirichlet)')
axes[1].semilogy(history['neumann'], 'm-', linewidth=1.2, alpha=0.8, label='Neumann')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss (log scale)', fontsize=12)
axes[1].set_title('Loss Components', fontsize=13)
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

print("All loss components should decrease together.")
print("If BC loss stays high, increase its weight.")

---

## Part 8: Visualize Predicted Head Distribution

Now we evaluate the trained PINN on a dense grid to visualize the hydraulic head field.

In [ ]:
# ───────────────────────────────────────────────────────
# PREDICT HEAD FIELD ON A GRID
# ───────────────────────────────────────────────────────

nx, ny = 100, 50
x_grid = np.linspace(0, L, nx)
y_grid = np.linspace(0, D, ny)
X_mesh, Y_mesh = np.meshgrid(x_grid, y_grid)

xy_grid = torch.tensor(
    np.column_stack([X_mesh.ravel(), Y_mesh.ravel()]),
    dtype=torch.float32
).to(device)

model.eval()
with torch.no_grad():
    h_pred = model(xy_grid).cpu().numpy().reshape(ny, nx)

# Plot head contours
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Filled contour
cf = axes[0].contourf(X_mesh, Y_mesh, h_pred, levels=20, cmap='RdYlBu_r')
plt.colorbar(cf, ax=axes[0], label='Hydraulic Head h (m)')
axes[0].set_xlabel('x (m)', fontsize=12)
axes[0].set_ylabel('y (m)', fontsize=12)
axes[0].set_title('PINN Predicted Head Distribution', fontsize=13)
axes[0].set_aspect('equal')

# Contour lines (equipotential lines)
cs = axes[1].contour(X_mesh, Y_mesh, h_pred, levels=15, cmap='RdYlBu_r')
axes[1].clabel(cs, inline=True, fontsize=8, fmt='%.1f')
axes[1].set_xlabel('x (m)', fontsize=12)
axes[1].set_ylabel('y (m)', fontsize=12)
axes[1].set_title('Equipotential Lines (Head Contours)', fontsize=13)
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()

print(f"Head ranges from {h_pred.min():.2f} m to {h_pred.max():.2f} m")
print(f"Expected: {h_downstream} m to {h_upstream} m")

---

## Part 9: Compare with Analytical Solution

For a simple rectangular domain with linear BCs on top and Dirichlet on sides, the analytical
solution (1D steady-state approximation) is:

$$h_{\text{analytical}}(x) = h_{\text{upstream}} + (h_{\text{downstream}} - h_{\text{upstream}}) \cdot \frac{x}{L}$$

In [ ]:
# ───────────────────────────────────────────────────────
# COMPARISON WITH ANALYTICAL SOLUTION
# ───────────────────────────────────────────────────────

# Analytical 1D solution: h(x) = h_up + (h_down - h_up) * x / L
h_analytical = h_upstream + (h_downstream - h_upstream) * X_mesh / L

# Error field
error = np.abs(h_pred - h_analytical)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# PINN solution
cf1 = axes[0].contourf(X_mesh, Y_mesh, h_pred, levels=20, cmap='RdYlBu_r')
plt.colorbar(cf1, ax=axes[0], label='h (m)')
axes[0].set_title('PINN Solution', fontsize=13)
axes[0].set_xlabel('x (m)')
axes[0].set_ylabel('y (m)')
axes[0].set_aspect('equal')

# Analytical solution
cf2 = axes[1].contourf(X_mesh, Y_mesh, h_analytical, levels=20, cmap='RdYlBu_r')
plt.colorbar(cf2, ax=axes[1], label='h (m)')
axes[1].set_title('Analytical Solution', fontsize=13)
axes[1].set_xlabel('x (m)')
axes[1].set_ylabel('y (m)')
axes[1].set_aspect('equal')

# Absolute error
cf3 = axes[2].contourf(X_mesh, Y_mesh, error, levels=20, cmap='Reds')
plt.colorbar(cf3, ax=axes[2], label='|Error| (m)')
axes[2].set_title('Absolute Error', fontsize=13)
axes[2].set_xlabel('x (m)')
axes[2].set_ylabel('y (m)')
axes[2].set_aspect('equal')

plt.tight_layout()
plt.show()

# Quantitative comparison
rmse = np.sqrt(np.mean(error ** 2))
max_err = error.max()
mean_err = error.mean()
print(f"\nComparison with analytical solution:")
print(f"  RMSE:      {rmse:.4f} m")
print(f"  Max error: {max_err:.4f} m")
print(f"  Mean error: {mean_err:.4f} m")
print(f"  Relative error: {rmse / (h_upstream - h_downstream) * 100:.2f}%")

In [ ]:
# ───────────────────────────────────────────────────────
# HEAD PROFILES AT SPECIFIC DEPTHS
# ───────────────────────────────────────────────────────

fig, ax = plt.subplots(1, 1, figsize=(10, 5))

depths_to_plot = [0.0, D/4, D/2, 3*D/4, D]
colors = plt.cm.viridis(np.linspace(0, 1, len(depths_to_plot)))

for depth_val, color in zip(depths_to_plot, colors):
    # PINN prediction at this depth
    x_line = torch.tensor(
        np.column_stack([x_grid, np.full_like(x_grid, depth_val)]),
        dtype=torch.float32
    ).to(device)
    
    with torch.no_grad():
        h_line = model(x_line).cpu().numpy().flatten()
    
    ax.plot(x_grid, h_line, '-', color=color, linewidth=2,
            label=f'PINN y={depth_val:.1f}m')

# Analytical
h_anal_line = h_upstream + (h_downstream - h_upstream) * x_grid / L
ax.plot(x_grid, h_anal_line, 'k--', linewidth=2, label='Analytical (1D)')

ax.set_xlabel('x (m)', fontsize=12)
ax.set_ylabel('Hydraulic Head h (m)', fontsize=12)
ax.set_title('Head Profiles at Different Depths', fontsize=13)
ax.legend(fontsize=9, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Profiles at all depths are close to the analytical 1D solution.")
print("The PINN naturally satisfies the 2D Laplace equation.")

---

## Practice Exercises

### Exercise 1: Change Boundary Conditions
Modify the upstream head to 15m and downstream to 0m. How does the head field change?

### Exercise 2: Add a Cutoff Wall
Add extra boundary points along x=5m, y=[0, 2.5]m with h=6m (simulating a cutoff wall). How does this affect flow?

### Exercise 3: Network Size Sensitivity
Try `hidden_size=32` vs `hidden_size=128`. Compare accuracy and training time.

In [ ]:
# EXERCISE SPACE
# Try the exercises above here

# Exercise 1: Change h_upstream = 15.0, h_downstream = 0.0
# Re-generate boundary points and retrain

# Exercise 2: Add cutoff wall boundary points
# x_cutoff = torch.full((50, 1), 5.0)
# y_cutoff = torch.rand(50, 1) * 2.5
# Append to xy_bc and h_bc

# Exercise 3: Compare architectures
# model_small = SeepagePINN(hidden_size=32)
# model_large = SeepagePINN(hidden_size=128)

---

## Key Takeaways

1. **PINNs encode physical laws directly in the loss function.** Instead of needing labeled data, they enforce PDEs (Laplace, Darcy, consolidation) at random collocation points.

2. **Automatic differentiation computes PDE derivatives exactly.** PyTorch's autograd gives us partial derivatives through the network without finite differences.

3. **Boundary conditions act as the training data.** Dirichlet BCs (fixed head) and Neumann BCs (zero flux) guide the solution just like physical constraints.

4. **Loss weighting matters.** The balance between physics loss and BC loss controls solution accuracy. BC weight is typically higher to ensure physical boundaries are respected.

5. **PINNs are mesh-free.** Unlike FEM, PINNs do not need structured meshes, making them attractive for irregular geotechnical domains.

---

## How This Connects to ML

| PINN Concept | Traditional ML Equivalent |
|---|---|
| Physics loss | Regularization (but from physics, not statistics) |
| Collocation points | Training samples (but generated, not measured) |
| Boundary conditions | Labeled data at known locations |
| PDE residual minimization | Fitting to target values |
| Domain knowledge in loss | Feature engineering in classical ML |

**PINNs bridge physics and ML** — tomorrow we explore Graph Neural Networks for soil particle and mesh networks.

---

*Day 078 of 100 — Teaching neural networks the laws of physics!*